In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab / Linux environment)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 3-Class Disaster Triage: Neighbourhood-Based (NB) Undersampling + Balanced Random Forest (`models/train_custom_overlap.ipynb`)

Loads the full emergency cohort from **`datasets/5v_cleandf.RData`** (~558,000 visits with valid ESI) and executes the **Neighbourhood-Based (NB) Undersampling** pipeline ported directly from the research repository [`NB-undersampling/`](file:///home/apt2736/PKM_RF/NB-undersampling/):
1. **Adaptive Radius Formulation**: $k = \text{round}\left(\sqrt{\text{IR}} + \sqrt{N_{\text{train}}}\right)$ dynamically scales neighbourhood size to match imbalance severity.
2. **Mutual Neighbourhood Denoising**: Removes invasive majority samples forming mutual Tomek links with minority resuscitations while strictly preserving **100% of all RED (ESI 1)** cases.
3. **Balanced Random Forest Classification**: Fits `RandomForestClassifier(n_estimators=300, max_depth=14, class_weight='balanced')` on the cleaned manifold.
4. **Holdout Test Set Evaluation**: Reports per-tier and macro **Recall (Sensitivity)**, **Specificity (True Negative Rate)**, **Balanced Accuracy**, and **ROC-AUC (One-vs-Rest)**.

```mermaid
flowchart TD
    Raw["Raw 8 Arrival Features X in R^8 (558,029 Visits)"] --> Imputer["SimpleImputer(strategy='median') & StandardScaler()"]
    Imputer --> NB["NB-Undersampling (Adaptive k = sqrt(IR) + sqrt(N))"]
    NB --> RF["Balanced Random Forest Classifier (n_estimators=300, class_weight='balanced')"]
    RF --> Eval["Holdout Test Evaluation: Recall, Specificity, Balanced Accuracy, ROC-AUC"]
    RF --> Viz["3x3 Confusion Matrix & Feature Importance Plots"]
```

### 🎯 3-Tier Disaster Triage Acuity Mapping
1. **`Tier 0: RED (ESI 1)`** ($y=0$): Immediate Resuscitation / Life Threat ($5,271$ visits, $\sim 0.94\%$).
2. **`Tier 1: YELLOW (ESI 2–3)`** ($y=1$): Emergent & Urgent conditions ($440,059$ visits, $\sim 78.86\%$).
3. **`Tier 2: GREEN (ESI 4–5)`** ($y=2$): Semi-urgent & Non-urgent conditions ($112,699$ visits, $\sim 20.19\%$).

### 📐 Evaluation Metrics Formulation
- **Recall (Sensitivity) per Class $c$**: $\text{Recall}_c = \frac{\text{TP}_c}{\text{TP}_c + \text{FN}_c}$
- **Specificity (True Negative Rate) per Class $c$**: $\text{Specificity}_c = \frac{\text{TN}_c}{\text{TN}_c + \text{FP}_c}$
- **Macro Balanced Accuracy**: $\text{Balanced Acc} = \frac{1}{3}\sum_{c=0}^{2} \text{Recall}_c$
- **Multiclass ROC-AUC**: One-vs-Rest (OvR) area under the ROC curve for each tier and macro-average.

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData (8 Core Triage Features, All Non-NA ESI Rows Kept)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

candidate_paths <- c(
  "../datasets/5v_cleandf.RData",
  "datasets/5v_cleandf.RData",
  "/kaggle/working/PKM_RF/datasets/5v_cleandf.RData",
  "/kaggle/input/5v-cleandf/5v_cleandf.RData",
  "/kaggle/input/disaster-triage-dataset/5v_cleandf.RData",
  "/kaggle/input/5v-raw/5v_cleandf.RData"
)

data_file <- NULL
for (p in candidate_paths) {
  if (file.exists(p)) {
    data_file <- p
    break
  }
}

if (is.null(data_file)) {
  stop("Could not find 5v_cleandf.RData in any candidate paths!")
}

cat(sprintf("Loading RData from: %s ...\n", data_file))
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

# Filter ONLY rows where ESI is not NA (retaining all ~558k observations)
valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

# Construct dataframe for 8 core triage features + ESI
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  esi                     = as.numeric(raw_esi_char)
)

feature_cols <- c(
  "age", "cc_breathingdifficulty", "gender",
  "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr", "triage_vital_o2"
)

# Export matrices to Python (NAs preserved for SimpleImputer)
raw_mat_export <- as.matrix(df_master[, feature_cols])
esi_export     <- as.numeric(df_master$esi)

cat(sprintf("Exported Full Dataset to Python: %d rows, %d feature columns\n",
            nrow(raw_mat_export), ncol(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Retrieve Data, Partition (70/15/15 Stratified) & Impute + Scale
# ---------------------------------------------------------------------------
import os, json, pickle, warnings, time
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

# Retrieve matrices from R environment with fallback to pyreadr
try:
    from rpy2.robjects import r
    raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
    esi_all     = np.array(r('esi_export'), dtype=np.int32)
except Exception:
    import pyreadr
    candidate_paths = [
        '../datasets/5v_cleandf.RData',
        'datasets/5v_cleandf.RData',
        '/kaggle/working/PKM_RF/datasets/5v_cleandf.RData',
        '/kaggle/input/5v-cleandf/5v_cleandf.RData'
    ]
    rdata_path = next(p for p in candidate_paths if os.path.exists(p))
    res = pyreadr.read_r(rdata_path)
    df_raw = res[list(res.keys())[0]]
    df_raw = df_raw[df_raw['esi'].notna()]
    gender = df_raw['gender'].apply(lambda x: 1 if str(x) == 'Male' else (0 if str(x) == 'Female' else np.nan)).values
    cc_bd = df_raw['cc_breathingdifficulty'].values if 'cc_breathingdifficulty' in df_raw.columns else np.full(len(df_raw), np.nan)
    raw_mat_all = np.column_stack([
        df_raw['age'].values,
        cc_bd,
        gender,
        df_raw['triage_vital_hr'].values,
        df_raw['triage_vital_sbp'].values,
        df_raw['triage_vital_dbp'].values,
        df_raw['triage_vital_rr'].values,
        df_raw['triage_vital_o2'].values
    ]).astype(np.float64)
    esi_all = df_raw['esi'].astype(int).values

FEATURES = [
    'age', 'cc_breathingdifficulty', 'gender',
    'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp',
    'triage_vital_rr', 'triage_vital_o2'
]

# Construct 3-Tier Target Variable:
# 0 = RED (ESI 1), 1 = YELLOW (ESI 2-3), 2 = GREEN (ESI 4-5)
y_all = np.zeros(len(esi_all), dtype=np.int32)
y_all[esi_all == 1] = 0
y_all[np.isin(esi_all, [2, 3])] = 1
y_all[np.isin(esi_all, [4, 5])] = 2
TIER_LABELS = ['RED (ESI 1)', 'YELLOW (ESI 2-3)', 'GREEN (ESI 4-5)']

print("=" * 75)
print("  3-TIER DISASTER TRIAGE COHORT (5v_cleandf.RData)")
print("=" * 75)
print(f"Total Valid ESI Encounters: {len(esi_all):,}")
for c, lbl in enumerate(TIER_LABELS):
    count_c = np.sum(y_all == c)
    print(f"  * Tier {c} [{lbl:<17}]: {count_c:>7,} ({count_c/len(y_all)*100:.2f}%)")
print("=" * 75 + "\n")

# Stratified 70% Train / 15% Validation / 15% Test Split
itr, itmp = train_test_split(np.arange(len(esi_all)), test_size=0.30, stratify=y_all, random_state=42)
iva, ite  = train_test_split(itmp, test_size=0.50, stratify=y_all[itmp], random_state=42)

y_train, y_val, y_test = y_all[itr], y_all[iva], y_all[ite]

# Median Imputation + Standard Scaling fitted strictly on Training partition
imputer   = SimpleImputer(strategy='median')
X_tr_imp  = imputer.fit_transform(raw_mat_all[itr])
X_val_imp = imputer.transform(raw_mat_all[iva])
X_te_imp  = imputer.transform(raw_mat_all[ite])

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_tr_imp)
X_val   = scaler.transform(X_val_imp)
X_test  = scaler.transform(X_te_imp)

print(f"Partition Dimensions:")
print(f"  * Training Set  : {X_train.shape[0]:,} visits (70%)")
print(f"  * Validation Set: {X_val.shape[0]:,} visits (15%)")
print(f"  * Holdout Test  : {X_test.shape[0]:,} visits (15%)")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Neighbourhood-Based (NB) Undersampling Algorithm Suite
#
# Ported faithfully from NB-undersampling/*.R:
#   - NB-Tomek (searchKnn_UnderS_Tomek.R)
#   - NB-Comm  (searchKnn_UnderS_Common.R)
#   - NB-Basic (searchKnn_UnderS.R)
#
# Adaptive Radius Formula:
#   k = round(sqrt(Imbalance_Ratio) + sqrt(N_samples))
# ---------------------------------------------------------------------------

def nb_tomek_undersample(X, y, minority_class=0, batch_size=25000, n_jobs=-1):
    """
    Neighbourhood-Based Tomek (NB-Tomek) Undersampling.
    Ported from NB-undersampling/searchKnn_UnderS_Tomek.R.

    Algorithm:
      1. Computes adaptive neighbourhood size: k = round(sqrt(IR) + sqrt(N))
      2. Finds k-nearest neighbours for all instances.
      3. For each majority instance, if a mutual k-NN link exists with any minority
         instance (Tomek link), the majority instance is eliminated.
      4. Preserves 100% of minority instances.
    """
    minority_mask = (y == minority_class)
    majority_mask = ~minority_mask

    pos_idx = np.where(minority_mask)[0]
    neg_idx = np.where(majority_mask)[0]
    num_pos = len(pos_idx)
    num_neg = len(neg_idx)

    if num_pos == 0 or num_neg == 0:
        return X, y, np.arange(len(y)), 0

    IR = float(num_neg) / float(num_pos)
    k = int(np.round(np.sqrt(IR) + np.sqrt(len(y))))
    k = max(3, min(k, len(y) - 1))

    print(f"  [NB-Tomek] Minority Class={minority_class} | IR={IR:.2f} | Adaptive k={k} | Total N={len(y):,}")

    # Order dataset: minority first (0 to num_pos-1), then majority (num_pos to N-1)
    ordered_indices = np.concatenate([pos_idx, neg_idx])
    X_ordered = X[ordered_indices]

    # Fit k-NN tree (k+1 to exclude self)
    nn = NearestNeighbors(n_neighbors=k + 1, algorithm='auto', n_jobs=n_jobs)
    nn.fit(X_ordered)

    # 1. Query k-NN for all minority instances
    pos_kneighbors = nn.kneighbors(X_ordered[:num_pos], return_distance=False)[:, 1:]
    pos_neighbor_sets = [set(pos_kneighbors[p]) for p in range(num_pos)]

    # 2. Query k-NN for majority instances in memory-safe batches
    keep_neg = np.ones(num_neg, dtype=bool)

    for b_start in range(0, num_neg, batch_size):
        b_end = min(b_start + batch_size, num_neg)
        batch_local_indices = np.arange(num_pos + b_start, num_pos + b_end)
        batch_X = X_ordered[batch_local_indices]
        batch_nn = nn.kneighbors(batch_X, return_distance=False)[:, 1:]

        for row_i in range(len(batch_local_indices)):
            global_local_idx = batch_local_indices[row_i]
            neg_offset = b_start + row_i
            neighbours = batch_nn[row_i]

            for m in range(k):
                nn_idx = neighbours[m]
                if nn_idx < num_pos:  # Neighbour is a minority instance
                    # Mutual link check: is this majority sample in minority's k-NN?
                    if global_local_idx in pos_neighbor_sets[nn_idx]:
                        keep_neg[neg_offset] = False
                        break

    kept_neg_original = neg_idx[keep_neg]
    kept_global_indices = np.concatenate([pos_idx, kept_neg_original])
    n_removed = int(np.sum(~keep_neg))

    print(f"  [NB-Tomek] Filtered out {n_removed:,} invasive majority boundary instances ({n_removed/num_neg*100:.2f}% pruned)")
    return X[kept_global_indices], y[kept_global_indices], kept_global_indices, n_removed


def nb_comm_undersample(X, y, minority_class=0, threshold=2, n_jobs=-1):
    """
    Neighbourhood-Based Common (NB-Comm) Undersampling.
    Ported from NB-undersampling/searchKnn_UnderS_Common.R.

    Algorithm:
      1. Computes adaptive neighbourhood size: k = round(sqrt(IR) + sqrt(N))
      2. Finds k-nearest neighbours of all minority instances.
      3. Counts how often each majority instance invades minority neighbourhoods.
      4. Prunes majority instances that appear >= threshold (default 2) times.
      5. Preserves 100% of minority instances.
    """
    minority_mask = (y == minority_class)
    majority_mask = ~minority_mask

    pos_idx = np.where(minority_mask)[0]
    neg_idx = np.where(majority_mask)[0]
    num_pos = len(pos_idx)
    num_neg = len(neg_idx)

    if num_pos == 0 or num_neg == 0:
        return X, y, np.arange(len(y)), 0

    IR = float(num_neg) / float(num_pos)
    k = int(np.round(np.sqrt(IR) + np.sqrt(len(y))))
    k = max(3, min(k, len(y) - 1))

    print(f"  [NB-Comm] Minority Class={minority_class} | IR={IR:.2f} | Adaptive k={k} | Total N={len(y):,}")

    ordered_indices = np.concatenate([pos_idx, neg_idx])
    X_ordered = X[ordered_indices]

    nn = NearestNeighbors(n_neighbors=k + 1, algorithm='auto', n_jobs=n_jobs)
    nn.fit(X_ordered)

    # Query only the minority instances
    pos_kneighbors = nn.kneighbors(X_ordered[:num_pos], return_distance=False)[:, 1:]

    neg_freq = np.zeros(num_neg, dtype=np.int32)
    for p in range(num_pos):
        for m in range(k):
            nn_idx = pos_kneighbors[p, m]
            if nn_idx >= num_pos:  # Neighbour is a majority instance
                neg_offset = nn_idx - num_pos
                neg_freq[neg_offset] += 1

    keep_neg = (neg_freq < threshold)
    kept_neg_original = neg_idx[keep_neg]
    kept_global_indices = np.concatenate([pos_idx, kept_neg_original])
    n_removed = int(np.sum(~keep_neg))

    print(f"  [NB-Comm] Filtered out {n_removed:,} invasive majority boundary instances ({n_removed/num_neg*100:.2f}% pruned)")
    return X[kept_global_indices], y[kept_global_indices], kept_global_indices, n_removed


print("✓ NB-Undersampling module successfully initialized.")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Apply NB-Undersampling to Clean Overlapping Boundary Noise
#
# 3-Class Strategy:
#   Stage 1: Clean invasive YELLOW + GREEN boundary noise around RED (ESI 1).
#            Preserves 100% of all RED resuscitations.
#   Stage 2: Clean invasive YELLOW boundary noise around GREEN (ESI 4-5).
#            Preserves 100% of GREEN semi-urgent visits.
# ---------------------------------------------------------------------------
print("=" * 75)
print("  EXECUTING MULTI-TIER NB-UNDERSAMPLING CLEANUP")
print("=" * 75)

print(f"\nOriginal Training Partition: {len(y_train):,} encounters")
for c, lbl in enumerate(TIER_LABELS):
    print(f"  * {lbl:<17}: {np.sum(y_train == c):>7,} visits")

t_start = time.time()

# --- Stage 1: Clean majority noise around RED (ESI 1, class 0) ---
print("\n--- Stage 1: NB-Tomek Boundary Denoising around RED (ESI 1) ---")
X_s1, y_s1, kept_idx_s1, rem_s1 = nb_tomek_undersample(
    X_train, y_train, minority_class=0, batch_size=25000, n_jobs=-1
)

# --- Stage 2: Clean YELLOW noise around GREEN (ESI 4-5, class 2) ---
print("\n--- Stage 2: NB-Tomek Boundary Denoising between YELLOW and GREEN ---")
# Filter to YELLOW (1) and GREEN (2) subset from Stage 1
yg_mask = np.isin(y_s1, [1, 2])
yg_local_indices = np.where(yg_mask)[0]
red_local_indices = np.where(y_s1 == 0)[0]

X_yg = X_s1[yg_local_indices]
y_yg = y_s1[yg_local_indices]

# Clean YELLOW (majority in this subset) vs GREEN (minority in this subset, class 2)
_, _, kept_yg_sub_indices, rem_s2 = nb_tomek_undersample(
    X_yg, y_yg, minority_class=2, batch_size=25000, n_jobs=-1
)

# Combine kept RED instances + kept YG instances
final_s1_indices = np.concatenate([red_local_indices, yg_local_indices[kept_yg_sub_indices]])
X_train_clean = X_s1[final_s1_indices]
y_train_clean = y_s1[final_s1_indices]

elapsed = time.time() - t_start
print(f"\n✓ Multi-Tier NB-Undersampling Completed in {elapsed:.1f}s")
print(f"Total Instances Pruned: {len(y_train) - len(y_train_clean):,} boundary noise samples\n")

print("=" * 75)
print("  CLEANED TRAINING COHORT DISTRIBUTION")
print("=" * 75)
for c, lbl in enumerate(TIER_LABELS):
    orig = np.sum(y_train == c)
    clean = np.sum(y_train_clean == c)
    ret_pct = clean / orig * 100
    print(f"  * {lbl:<17}: {clean:>7,} / {orig:>7,} ({ret_pct:.2f}% retained)")
print("=" * 75)

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Fit Balanced Random Forest on Cleaned Training Cohort
# ---------------------------------------------------------------------------
print("Training Balanced Random Forest Classifier on NB-Denoised Training Cohort...")
print("  Hyperparameters: n_estimators=300, max_depth=14, min_samples_split=20, class_weight='balanced'")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight=None,
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
rf_model.fit(X_train_clean, y_train_clean)
print(f"✓ Model Training Completed in {time.time()-t0:.1f}s!")

# Validation Check
pred_val = rf_model.predict(X_val)
val_bacc = balanced_accuracy_score(y_val, pred_val)
val_rec0 = recall_score(y_val, pred_val, labels=[0], average=None, zero_division=0)[0]
print(f"\nValidation Set Quick Check:")
print(f"  * Macro Balanced Accuracy : {val_bacc*100:.2f}%")
print(f"  * RED (ESI 1) Recall      : {val_rec0*100:.2f}%")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: Full Holdout Test Evaluation — Recall, Specificity, Balanced Acc, ROC-AUC
# ---------------------------------------------------------------------------
pred_test = rf_model.predict(X_test)
p_test    = rf_model.predict_proba(X_test)

# 1. Overall & Macro Metrics
acc         = accuracy_score(y_test, pred_test)
bal_acc     = balanced_accuracy_score(y_test, pred_test)
macro_f1    = f1_score(y_test, pred_test, average='macro', zero_division=0)
weighted_f1 = f1_score(y_test, pred_test, average='weighted', zero_division=0)

# 2. Per-Class Recall (Sensitivity), Precision, F1
recall_per = recall_score(y_test, pred_test, average=None, zero_division=0)
prec_per   = precision_score(y_test, pred_test, average=None, zero_division=0)
f1_per     = f1_score(y_test, pred_test, average=None, zero_division=0)

# 3. Per-Class Specificity (True Negative Rate)
cm = confusion_matrix(y_test, pred_test, labels=[0, 1, 2])
specificity_per = []
for c in range(3):
    tp = cm[c, c]
    fn = cm[c, :].sum() - tp
    fp = cm[:, c].sum() - tp
    tn = cm.sum() - tp - fn - fp
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    specificity_per.append(spec)
macro_spec = np.mean(specificity_per)

# 4. Per-Class & Macro ROC-AUC (One-vs-Rest)
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
roc_auc_ovr = roc_auc_score(y_test_bin, p_test, average='macro', multi_class='ovr')
roc_auc_per = []
for c in range(3):
    auc_c = roc_auc_score(y_test_bin[:, c], p_test[:, c])
    roc_auc_per.append(auc_c)

# Construct Detailed Summary DataFrame
report_rows = []
for c, lbl in enumerate(TIER_LABELS):
    report_rows.append({
        'Triage_Tier': lbl,
        'True_Visits': int(np.sum(y_test == c)),
        'Predicted_Visits': int(np.sum(pred_test == c)),
        'Recall (Sensitivity)': f"{recall_per[c]*100:.2f}%",
        'Specificity': f"{specificity_per[c]*100:.2f}%",
        'Precision': f"{prec_per[c]*100:.2f}%",
        'F1_Score': round(f1_per[c], 4),
        'ROC_AUC (OvR)': round(roc_auc_per[c], 4)
    })

report_df = pd.DataFrame(report_rows)

print("=" * 105)
print("     HOLDOUT TEST EVALUATION: NB-UNDERSAMPLING + BALANCED RANDOM FOREST (3-TIER)")
print("=" * 105)
print(f"Total Test Encounters       : {len(y_test):,} visits")
print(f"Overall Accuracy            : {acc*100:.2f}%")
print(f"Macro Balanced Accuracy     : {bal_acc*100:.2f}%")
print(f"Macro Specificity           : {macro_spec*100:.2f}%")
print(f"Macro ROC-AUC (OvR)         : {roc_auc_ovr:.4f}")
print(f"Macro F1-Score              : {macro_f1:.4f}")
print(f"Weighted F1-Score           : {weighted_f1:.4f}")
print("-" * 105)
print(report_df.to_string(index=False))
print("=" * 105 + "\n")

print("Classification Report:")
print(classification_report(y_test, pred_test, target_names=TIER_LABELS, digits=4))

# Export Report CSV
reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'nb_undersampling_rf_3class_report.csv')
report_df.to_csv(report_file, index=False)
print(f"✓ Report successfully saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: 3x3 Confusion Matrix Heatmap
# ---------------------------------------------------------------------------
plots_dir = f'{ROOT}/plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'distant_analysis'), exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(8.5, 7))
annot = np.empty_like(cm, dtype=object)
for i in range(3):
    for j in range(3):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.2f}%)"

sns.heatmap(
    cm_norm, annot=annot, fmt='', cmap='Blues', cbar=True, ax=ax,
    vmin=0, vmax=1, xticklabels=TIER_LABELS, yticklabels=TIER_LABELS
)

ax.set_title(
    f'NB-Undersampling + Balanced Random Forest: 3-Class Confusion Matrix\n'
    f'Balanced Accuracy: {bal_acc*100:.2f}% | Macro ROC-AUC: {roc_auc_ovr:.4f}',
    fontsize=11.5, fontweight='bold', pad=12
)
ax.set_xlabel('Predicted Disaster Triage Tier', fontsize=11, fontweight='bold')
ax.set_ylabel('True Disaster Triage Tier', fontsize=11, fontweight='bold')

plt.tight_layout()
cm_path1 = os.path.join(plots_dir, 'distant_analysis', 'nb_undersampling_rf_3class_confusion_matrix.png')
cm_path2 = os.path.join(plots_dir, 'image', 'nb_undersampling_rf_3class_confusion_matrix.png')
plt.savefig(cm_path1, dpi=300, bbox_inches='tight')
plt.savefig(cm_path2, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Confusion Matrix saved to: {cm_path1}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: Feature Importance Bar Chart
# ---------------------------------------------------------------------------
importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.barplot(data=feat_imp_df, x='Importance', y='Feature', color='#1f77b4', ax=ax)
ax.set_title('NB-Undersampling + RF: Gini Feature Importance (8 Arrival Triage Features)', fontsize=12.5, fontweight='bold', pad=12)
ax.set_xlabel('Normalized Gini Importance', fontsize=11, fontweight='bold')
ax.set_ylabel('Feature', fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
imp_path = os.path.join(plots_dir, 'distant_analysis', 'nb_undersampling_rf_feature_importance.png')
plt.savefig(imp_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Feature importance plot saved to: {imp_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 9: Export Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'imputer': imputer,
    'scaler': scaler,
    'rf_model': rf_model,
    'features': FEATURES,
    'tier_labels': TIER_LABELS
}

bundle_file = os.path.join(deploy_dir, 'nb_undersampling_rf_3class_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='NB_Undersampling_Plus_Balanced_Random_Forest_3Class',
    nb_method='Neighbourhood-Based Tomek (NB-Tomek, k = round(sqrt(IR) + sqrt(N)))',
    nb_reference='NB-undersampling/searchKnn_UnderS_Tomek.R',
    classifier='RandomForestClassifier',
    n_estimators=int(rf_model.n_estimators),
    max_depth=int(rf_model.max_depth),
    class_weight='balanced',
    features=FEATURES,
    tier_labels=TIER_LABELS,
    total_samples=len(esi_all),
    holdout_test_samples=len(y_test),
    overall_accuracy=round(acc, 4),
    macro_balanced_accuracy=round(bal_acc, 4),
    macro_specificity=round(macro_spec, 4),
    macro_roc_auc_ovr=round(roc_auc_ovr, 4),
    macro_f1=round(macro_f1, 4),
    per_class_metrics=report_rows
)

manifest_file = os.path.join(deploy_dir, 'nb_undersampling_rf_3class_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Deployment Bundle  : {bundle_file}")
print(f"✓ Deployment Manifest: {manifest_file}")